# Lab 1 - Qualify a Live Model Router Candidate

> About 40 minutes. Prerequisite: Lab 0 complete and the repository-root `.env` populated.

## Scenario

Northstar Devices has launched its Aurora X1 phone. A carrier disruption, duplicate-payment incident, and phishing campaign have pushed support demand beyond the team's fixed-model capacity. Support Operations wants to use Model Router to **lower inference cost and latency without reducing policy compliance or account-security quality**.

You own the model-selection policy. Before this router can enter the three-arm promotion experiment in Lab 2, answer one question: **is the live router operationally credible enough to become a candidate?**

## Candidate contract

1. Invoke the active Model Router with 12 core cases and 8 harder challenge cases.
2. Capture the underlying model, token usage, latency, response, workflow, and risk tier.
3. Require every request to complete and every queue classification to be exact.
4. Require at least 80% overall quality passes and no failed high-risk cases.
5. Export an auditable scorecard for the change review.

Token usage is the transparent cost proxy in this smoke test because prices vary by model, region, and date. Lab 2 joins current pricing when comparing deployments.

> **Execution contract:** This notebook is live-only. It stops unless `AZURE_AI_PROJECT_ENDPOINT` and `MODEL_ROUTER_DEPLOYMENT` are configured and Microsoft Entra authentication succeeds. Running all cells invokes Azure resources and incurs usage.

The cases are synthetic and privacy-safe; model responses and operational measurements are live. Passing these smoke gates qualifies a candidate for deeper evaluation. It is not a production approval, benchmark, price quote, or service-level claim.

## Environment and authentication

Install the repository-root `requirements.txt`, select that environment as the notebook kernel, and configure the shared root `.env`:

- `AZURE_AI_PROJECT_ENDPOINT=https://<account>.services.ai.azure.com/api/projects/<project>`
- `BASELINE_GPT_DEPLOYMENT=baseline-gpt`
- `GPT_FAMILY_ROUTER_DEPLOYMENT=router-gpt-family`
- `OPEN_WEIGHT_ROUTER_DEPLOYMENT=router-open-weight`
- `MODEL_ROUTER_DEPLOYMENT=<active-router-deployment>`

`MODEL_ROUTER_DEPLOYMENT` selects the single router measured in this run. The other names remain available for the Lab 2 comparison.

Authentication uses `DefaultAzureCredential`; no key is stored in this notebook. For local execution, sign in with `az login`. If VS Code is signed into a different tenant, set `AZURE_TOKEN_CREDENTIALS=AzureCliCredential` before starting the kernel.

There is no simulation switch. A missing endpoint, deployment, credential, or live response fails the run.

In [ ]:
import json
import os
import re
import time
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
BASELINE_GPT_DEPLOYMENT = os.getenv("BASELINE_GPT_DEPLOYMENT", "baseline-gpt")
GPT_FAMILY_ROUTER_DEPLOYMENT = os.getenv("GPT_FAMILY_ROUTER_DEPLOYMENT", "router-gpt-family")
OPEN_WEIGHT_ROUTER_DEPLOYMENT = os.getenv("OPEN_WEIGHT_ROUTER_DEPLOYMENT", "router-open-weight")
ROUTER_DEPLOYMENT = os.getenv("MODEL_ROUTER_DEPLOYMENT", OPEN_WEIGHT_ROUTER_DEPLOYMENT)

if not PROJECT_ENDPOINT or not ROUTER_DEPLOYMENT:
    raise RuntimeError(
        "Set AZURE_AI_PROJECT_ENDPOINT and MODEL_ROUTER_DEPLOYMENT before running this live-only notebook."
    )

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 110})

print("Execution mode: LIVE MICROSOFT FOUNDRY MODEL ROUTER")
print(f"Active router deployment: {ROUTER_DEPLOYMENT}")
print(f"Configured baseline: {BASELINE_GPT_DEPLOYMENT}")
print(f"Configured GPT-family router: {GPT_FAMILY_ROUTER_DEPLOYMENT}")
print(f"Configured open-weight router: {OPEN_WEIGHT_ROUTER_DEPLOYMENT}")

## 1. Launch-incident evaluation workload

The workload models four support workflows created by the Aurora X1 launch incident:

| Workflow | Operational consequence of failure |
|---|---|
| Queue triage | Misroutes work and increases handling time |
| Policy answer | Gives an unsupported refund, return, disclosure, or recovery answer |
| Case handoff | Drops ownership, deadlines, verified facts, or security state |
| Remedy decision | Trades customer impact against policy and fraud controls incorrectly |

The 12-case **core dataset** covers expected launch traffic. The 8-case **challenge dataset** adds mixed-intent, negation, unverified claims, and pressure to bypass security controls. Eight cases are marked high risk so averages cannot hide policy or account-security failures.

For each live request the notebook captures the dataset split, workflow, risk tier, selected underlying model, service-reported tokens, end-to-end latency, response text, and local quality result.

Token usage is reported as the cost proxy. Join current official prices by selected model, region, and date before making a governed cost claim.

### Evaluation design

Classification requires one exact queue label. Policy answers must stay within the supplied rule. Handoffs must preserve requested facts, ownership, and security state. Remedy decisions must apply every supplied constraint without inventing an exception.

The core and challenge sets are reported separately. High-risk cases form a hard smoke gate: a strong average cannot compensate for restoring an unverified account, disclosing protected records, or omitting a required security escalation.

Reference overlap is transparent and reproducible, but remains a regression indicator. Production promotion also requires calibrated policy-compliance and safety evaluators, repeated samples, human review of critical failures, and dated price data.

In [ ]:
data_directories = [Path.cwd(), Path.cwd() / "lab-1-live-evaluation"]
data_directory = next(
    (directory for directory in data_directories if (directory / "router_eval.jsonl").exists()),
    None,
)
if data_directory is None:
    raise FileNotFoundError("Run the notebook from the repository root or lab-1-live-evaluation directory.")


def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]


core_records = load_jsonl(data_directory / "router_eval.jsonl")
challenge_records = load_jsonl(data_directory / "router_eval_challenge.jsonl")
dataset = pd.DataFrame(core_records + challenge_records)

required_columns = {
    "request_id", "dataset_split", "workflow", "risk_tier", "task", "prompt", "ground_truth",
}
if missing_columns := required_columns - set(dataset.columns):
    raise ValueError(f"Evaluation datasets are missing columns: {sorted(missing_columns)}")

assert len(dataset) == 20 and dataset.request_id.is_unique
assert dataset.dataset_split.value_counts().to_dict() == {"core": 12, "challenge": 8}
assert dataset.groupby("task").size().to_dict() == {
    "classification": 5, "reasoning": 5, "retrieval": 5, "summarisation": 5,
}
assert dataset.risk_tier.value_counts().to_dict() == {"standard": 12, "high": 8}

display(dataset[[
    "request_id", "dataset_split", "workflow", "risk_tier", "task", "prompt", "ground_truth",
]])
print(f"Live evaluation rows: {len(dataset)} (12 core + 8 challenge)")

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from openai import APIConnectionError, APIStatusError


def create_response_with_retry(client, model, prompt, max_attempts=4):
    for attempt in range(1, max_attempts + 1):
        try:
            return client.responses.create(model=model, input=prompt)
        except (APIConnectionError, APIStatusError) as error:
            status_code = getattr(error, "status_code", None)
            retryable = status_code is None or status_code in {408, 409, 429} or status_code >= 500
            if not retryable or attempt == max_attempts:
                raise
            delay_seconds = min(2 ** (attempt - 1), 8)
            print(
                f"Transient Model Router error for attempt {attempt}/{max_attempts} "
                f"(status={status_code}); retrying in {delay_seconds}s."
            )
            time.sleep(delay_seconds)


live_rows = []
with (
    DefaultAzureCredential(process_timeout=60) as credential,
    AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    for item in dataset.itertuples(index=False):
        started = time.perf_counter()
        response = create_response_with_retry(openai_client, ROUTER_DEPLOYMENT, item.prompt)
        usage = getattr(response, "usage", None)
        live_rows.append({
            "request_id": item.request_id,
            "dataset_split": item.dataset_split,
            "workflow": item.workflow,
            "risk_tier": item.risk_tier,
            "task": item.task,
            "ground_truth": item.ground_truth,
            "response": getattr(response, "output_text", "") or "",
            "selected_model": getattr(response, "model", "unknown"),
            "input_tokens": getattr(usage, "input_tokens", None),
            "output_tokens": getattr(usage, "output_tokens", None),
            "latency_ms": (time.perf_counter() - started) * 1000,
        })
        print(
            f"{item.request_id}: {live_rows[-1]['selected_model']} | "
            f"{live_rows[-1]['latency_ms']:.0f} ms"
        )

live_results = pd.DataFrame(live_rows)
if len(live_results) != len(dataset) or live_results.response.eq("").any():
    raise RuntimeError("Every dataset row must produce a non-empty live Model Router response.")
if live_results[["input_tokens", "output_tokens"]].isna().any().any():
    raise RuntimeError("Token usage was missing from one or more live responses.")

print(f"Completed {len(live_results)} live Model Router calls.")

In [ ]:
def terms(value):

    return re.findall(r"[a-z0-9]+", str(value).lower())



def overlap_metrics(response, ground_truth):

    response_terms = terms(response)

    truth_terms = terms(ground_truth)

    response_counts = {term: response_terms.count(term) for term in set(response_terms)}

    truth_counts = {term: truth_terms.count(term) for term in set(truth_terms)}

    overlap = sum(min(response_counts.get(term, 0), truth_counts.get(term, 0)) for term in truth_counts)

    precision = overlap / max(1, len(response_terms))

    recall = overlap / max(1, len(truth_terms))

    f1 = 2 * precision * recall / max(1e-12, precision + recall)

    coverage = len(set(response_terms) & set(truth_terms)) / max(1, len(set(truth_terms)))

    return precision, recall, f1, coverage



metric_rows = live_results.apply(

    lambda row: overlap_metrics(row.response, row.ground_truth), axis=1, result_type="expand"

)

metric_rows.columns = ["precision", "recall", "f1", "reference_coverage"]

live_results = pd.concat([live_results, metric_rows], axis=1)

live_results["exact_match"] = (

    live_results.response.str.strip().str.lower() == live_results.ground_truth.str.strip().str.lower()

)

live_results["quality_pass"] = np.where(

    live_results.task.eq("classification"), live_results.exact_match, live_results.f1.ge(0.50)

)



display(live_results[[

    "request_id", "task", "selected_model", "input_tokens", "output_tokens",

    "latency_ms", "f1", "reference_coverage", "quality_pass"

]].round(3))



assert live_results.request_id.is_unique

print("All displayed measurements come from live Model Router responses.")

## 2. Live routing and candidate scorecard

The following summaries use only `live_results`, created from the 20 live service responses above. No latency, token, model-selection, or quality value is simulated. Results are sliced by core versus challenge traffic and by standard versus high risk so an overall average cannot conceal an operationally important failure.

In [ ]:
overall_summary = pd.Series({
    "live_requests": len(live_results),
    "underlying_models_observed": live_results.selected_model.nunique(),
    "quality_pass_rate": live_results.quality_pass.mean(),
    "mean_f1": live_results.f1.mean(),
    "mean_reference_coverage": live_results.reference_coverage.mean(),
    "total_input_tokens": live_results.input_tokens.sum(),
    "total_output_tokens": live_results.output_tokens.sum(),
    "tokens_per_request": (live_results.input_tokens + live_results.output_tokens).mean(),
    "latency_p50_ms": live_results.latency_ms.quantile(0.50),
    "latency_p95_ms": live_results.latency_ms.quantile(0.95),
})

by_split = live_results.groupby("dataset_split").agg(
    requests=("request_id", "count"),
    quality_pass_rate=("quality_pass", "mean"),
    mean_f1=("f1", "mean"),
    total_tokens=("input_tokens", "sum"),
    latency_p50_ms=("latency_ms", "median"),
    latency_p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
).round(3)
by_split["total_tokens"] += live_results.groupby("dataset_split").output_tokens.sum()

by_risk = live_results.groupby("risk_tier").agg(
    requests=("request_id", "count"),
    quality_pass_rate=("quality_pass", "mean"),
    mean_f1=("f1", "mean"),
    latency_p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
).round(3)

by_task = live_results.groupby("task").agg(
    requests=("request_id", "count"),
    models_observed=("selected_model", "nunique"),
    quality_pass_rate=("quality_pass", "mean"),
    mean_f1=("f1", "mean"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    latency_p50_ms=("latency_ms", "median"),
    latency_p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
).round(3)

display(
    overall_summary.to_frame("observed_value").round(3),
    by_split,
    by_risk,
    by_task,
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.barplot(data=live_results, x="task", y="f1", hue="dataset_split", errorbar=None, ax=axes[0])
axes[0].axhline(0.50, color="firebrick", linestyle="--", label="Reference threshold")
axes[0].set_title("Observed quality by task and split")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(fontsize=8)

token_long = live_results.melt(
    id_vars=["request_id", "task"],
    value_vars=["input_tokens", "output_tokens"],
    var_name="token_type",
    value_name="tokens",
)
sns.barplot(data=token_long, x="task", y="tokens", hue="token_type", errorbar=None, ax=axes[1])
axes[1].set_title("Token cost proxy per request")
axes[1].tick_params(axis="x", rotation=25)

latency_points = pd.DataFrame({
    "percentile": ["p50", "p95"],
    "latency_ms": [live_results.latency_ms.quantile(0.50), live_results.latency_ms.quantile(0.95)],
})
sns.barplot(data=latency_points, x="percentile", y="latency_ms", ax=axes[2])
axes[2].axhline(8_000, color="firebrick", linestyle="--", label="Smoke budget")
axes[2].set_title("Observed end-to-end latency")
axes[2].legend()

plt.tight_layout()
plt.show()

print("These are point-in-time live observations from 20 requests, not service benchmarks.")

## 3. Observed Model Router selection

The application sends every case to one deployment. The `selected_model` field returned by the service identifies the underlying model selected for that request.

```mermaid
flowchart LR
    C[12 core cases] --> MR[Microsoft Foundry Model Router]
    H[8 challenge cases] --> MR
    MR --> M[Selected underlying model]
    MR --> R[Response and token usage]
    R --> Q[Policy and quality checks]
    R --> L[Latency measurement]
    M --> O[Routing by workflow and risk]
    Q --> G[Candidate gate]
    L --> G
```

The notebook does not infer or emulate routing decisions. It reports only models returned by the live service and treats high-risk policy failures as a hard stop.

In [ ]:
selection_counts = (

    live_results.groupby(["task", "selected_model"]).size().rename("requests").reset_index()

)

selection_share = selection_counts.copy()

selection_share["share_within_task"] = selection_share.groupby("task").requests.transform(

    lambda values: values / values.sum()

)

display(selection_share.sort_values(["task", "requests"], ascending=[True, False]).round(3))



fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=live_results, x="task", hue="selected_model", ax=axes[0])

axes[0].set_title("Live underlying-model selections")

axes[0].tick_params(axis="x", rotation=25)

axes[0].legend(fontsize=8)

sns.scatterplot(

    data=live_results, x="latency_ms", y="f1", hue="selected_model",

    style="task", size="output_tokens", sizes=(40, 250), ax=axes[1]

)

axes[1].set_title("Observed quality, latency, and output tokens")

plt.tight_layout()

plt.show()

## 4. Model evolution and governance



A single live run can reveal which underlying models were selected, but it cannot prove that one deployment mode or model subset is better than another. That requires separately versioned Model Router deployments, identical evaluation prompts, repeated samples, and controlled assignment.



Use the observed distribution below to answer operational questions:



- Did the router select more than one underlying model?

- Which task types were routed to each observed model?

- Are quality failures concentrated by task or selected model?

- Are token or latency outliers associated with a particular route?



Promotion decisions should require task-level quality floors, safety checks, sufficient samples, current pricing, and repeated latency measurements.

In [ ]:
model_task_summary = live_results.groupby(["selected_model", "task"]).agg(

    requests=("request_id", "count"),

    quality_pass_rate=("quality_pass", "mean"),

    mean_f1=("f1", "mean"),

    mean_input_tokens=("input_tokens", "mean"),

    mean_output_tokens=("output_tokens", "mean"),

    latency_p50_ms=("latency_ms", "median"),

).reset_index()

display(model_task_summary.round(3))



observed_models = sorted(live_results.selected_model.unique())

print("Underlying models observed in this live run:")

for model in observed_models:

    print(" -", model)

In [ ]:
mean_total_tokens = (live_results.input_tokens + live_results.output_tokens).mean()
latency_p95_ms = live_results.latency_ms.quantile(0.95)
high_risk_results = live_results.query("risk_tier == 'high'")
challenge_results = live_results.query("dataset_split == 'challenge'")

candidate_gates = pd.Series({
    "all_20_requests_completed": len(live_results) == len(dataset) == 20,
    "all_responses_non_empty": live_results.response.ne("").all(),
    "all_token_usage_present": live_results[["input_tokens", "output_tokens"]].notna().all().all(),
    "all_queue_classifications_exact": live_results.query("task == 'classification'").exact_match.all(),
    "overall_quality_pass_rate_at_least_80pct": live_results.quality_pass.mean() >= 0.80,
    "challenge_quality_pass_rate_at_least_75pct": challenge_results.quality_pass.mean() >= 0.75,
    "all_high_risk_cases_pass": high_risk_results.quality_pass.all(),
    "latency_p95_at_most_8_seconds": latency_p95_ms <= 8_000,
    "mean_total_tokens_at_most_300": mean_total_tokens <= 300,
})
candidate_qualified = bool(candidate_gates.all())

display(candidate_gates.to_frame("passed"))
print(f"Observed p95 latency: {latency_p95_ms:.0f} ms (smoke budget: 8,000 ms)")
print(f"Observed mean total tokens: {mean_total_tokens:.1f} (smoke budget: 300)")
print("CANDIDATE QUALIFICATION:", "PASS" if candidate_qualified else "HOLD")
print("Promotion remains NOT ASSESSED until Lab 2 compares cost, latency, quality, and safety against the baseline.")

## 5. Live quality analysis



Quality is evaluated against task-specific reference answers. Classification uses strict exact match; the other tasks use token precision, recall, F1, and unique reference-term coverage.



These local metrics are deliberately transparent. Before production use, add privacy-reviewed safety evaluation, groundedness checks, policy-compliance rules, calibrated model-graded evaluators, and human review for high-risk failures.



Routing fixes selection problems. Prompt changes fix instruction problems. Evaluator calibration fixes measurement problems. Fine-tuning should be considered only after repeated domain failures remain across a representative live dataset.

In [ ]:
quality_by_task = live_results.groupby("task").agg(

    requests=("request_id", "count"),

    exact_match_rate=("exact_match", "mean"),

    precision=("precision", "mean"),

    recall=("recall", "mean"),

    f1=("f1", "mean"),

    reference_coverage=("reference_coverage", "mean"),

    quality_pass_rate=("quality_pass", "mean"),

).round(3)

display(quality_by_task)



quality_long = live_results.melt(

    id_vars=["request_id", "task"],

    value_vars=["precision", "recall", "f1", "reference_coverage"],

    var_name="metric", value_name="score",

)

plt.figure(figsize=(12, 5))

sns.barplot(data=quality_long, x="task", y="score", hue="metric", errorbar=None)

plt.ylim(0, 1)

plt.title("Quality metrics from live Model Router responses")

plt.xticks(rotation=25)

plt.tight_layout()

plt.show()

In [ ]:
failures = live_results.loc[

    ~live_results.quality_pass,

    ["request_id", "task", "selected_model", "ground_truth", "response", "f1", "reference_coverage", "latency_ms"],

]

if failures.empty:

    print("No failures under the notebook's local quality rules.")

else:

    display(failures.round(3))

    print("Review each failure for prompt, routing, evaluator, or domain-knowledge causes.")



latency_outlier_threshold = live_results.latency_ms.quantile(0.95)

token_outlier_threshold = live_results.output_tokens.quantile(0.95)

outliers = live_results.loc[

    (live_results.latency_ms >= latency_outlier_threshold)

    | (live_results.output_tokens >= token_outlier_threshold),

    ["request_id", "task", "selected_model", "latency_ms", "output_tokens", "quality_pass"],

]

print("Observed p95 latency threshold (ms):", round(latency_outlier_threshold, 1))

display(outliers.round(3))

## 6. Iterate with live evidence



A production optimization loop should compare separately versioned Model Router deployments or configurations against the same governed dataset.



```mermaid

flowchart LR

    D[Versioned evaluation prompts] --> R[Run live router deployment]

    R --> M[Capture route, response, tokens, latency]

    M --> E[Evaluate quality and safety]

    E --> A[Analyze failures and outliers]

    A --> C[Change mode, subset, prompt, or evaluator]

    C --> V[Deploy candidate version]

    V --> R

```



This notebook evaluates one configured deployment. It does not fabricate candidate deployments or replay synthetic trajectories.

In [ ]:
run_manifest = {
    "scenario": "northstar-aurora-x1-launch-support",
    "execution_mode": "live",
    "router_deployment": ROUTER_DEPLOYMENT,
    "requests": int(len(live_results)),
    "dataset_splits": live_results.dataset_split.value_counts().sort_index().to_dict(),
    "workflows": live_results.workflow.value_counts().sort_index().to_dict(),
    "risk_tiers": live_results.risk_tier.value_counts().sort_index().to_dict(),
    "tasks": live_results.task.value_counts().sort_index().to_dict(),
    "observed_models": sorted(live_results.selected_model.unique()),
    "quality_pass_rate": float(live_results.quality_pass.mean()),
    "high_risk_pass_rate": float(high_risk_results.quality_pass.mean()),
    "challenge_pass_rate": float(challenge_results.quality_pass.mean()),
    "mean_f1": float(live_results.f1.mean()),
    "total_input_tokens": int(live_results.input_tokens.sum()),
    "total_output_tokens": int(live_results.output_tokens.sum()),
    "mean_total_tokens": float(mean_total_tokens),
    "latency_p50_ms": float(live_results.latency_ms.quantile(0.50)),
    "latency_p95_ms": float(latency_p95_ms),
    "candidate_qualified": candidate_qualified,
    "candidate_gates": {name: bool(value) for name, value in candidate_gates.items()},
}
print(json.dumps(run_manifest, indent=2))

assert run_manifest["execution_mode"] == "live"
assert run_manifest["requests"] == 20
assert run_manifest["dataset_splits"] == {"challenge": 8, "core": 12}
assert run_manifest["observed_models"]
assert live_results.response.ne("").all()

## 7. Production observability and governance

Persist deployment version, selected underlying model, workflow, risk tier, dataset split and version, input/output tokens, latency, quality and safety outcomes, retries, and error category. Do not log raw sensitive prompts or responses without an approved data-handling policy.

Use Microsoft Entra authentication and least-privilege RBAC. Govern eligible model publishers and deployments with Azure Policy, keep optimization and final holdout datasets separate, and store generated artifacts in access-controlled locations.

Operational gates should cover:

- zero failed or empty responses;
- task-level policy, security, and quality floors;
- challenge and high-risk slice performance;
- p50 and p95 latency from repeated samples;
- token usage joined to a dated official pricing snapshot;
- minimum sample size and practical improvement over a fixed baseline;
- canary monitoring and a tested rollback path.

This 20-request run can qualify a candidate for Lab 2. It cannot establish lower production cost or latency because it does not yet include the controlled fixed-model baseline.

In [ ]:
production_readiness = pd.Series({

    "live_router_invoked_for_every_row": len(live_results) == len(dataset),

    "underlying_model_recorded": live_results.selected_model.ne("unknown").all(),

    "token_usage_recorded": live_results[["input_tokens", "output_tokens"]].notna().all().all(),

    "latency_recorded": live_results.latency_ms.gt(0).all(),

    "quality_evaluated": live_results[["f1", "reference_coverage"]].notna().all().all(),

    "safety_evaluation_configured": False,

    "representative_production_sample": False,

    "deployment_comparison_completed": False,

})

display(production_readiness.to_frame("status"))



print("Functional live evaluation:", "PASS" if production_readiness.iloc[:5].all() else "FAIL")

print("Production promotion:", "NOT ASSESSED - add safety, representative volume, and candidate comparison.")

## 8. Deployment configuration context



The live endpoint used above is a Microsoft Foundry project endpoint, and every application request targets the configured Model Router deployment name. Deployment mode and eligible model subset are management-plane configuration, not application-side routing logic.



Validate current Model Router versions, modes, model availability, regional support, quota, and pricing in the official documentation before changing a deployment. Keep deployment configuration versioned and evaluate each candidate against the same governed dataset.



The code path used in this notebook is:



```python

response = openai_client.responses.create(model=ROUTER_DEPLOYMENT, input=prompt)

selected_underlying_model = response.model

```



The `selected_underlying_model` values displayed in this notebook are returned by the live service.

In [ ]:
response_examples = live_results[[

    "request_id", "task", "selected_model", "response", "input_tokens", "output_tokens", "latency_ms"

]].copy()

response_examples["latency_ms"] = response_examples.latency_ms.round(1)

display(response_examples)



print("Live route verification:")

print(" - deployment:", ROUTER_DEPLOYMENT)

print(" - requests:", len(response_examples))

print(" - selected models:", ", ".join(sorted(response_examples.selected_model.unique())))

## 9. Azure AI Evaluation SDK over live responses

The evaluation below passes the 20 already-captured live Model Router responses to the local `azure-ai-evaluation` SDK. It does not invoke a second application model or overwrite either public seed dataset.

`F1ScoreEvaluator` provides a standard lexical score. The custom reference-coverage evaluator provides a transparent companion metric. These signals support the smoke gate, but do not by themselves prove policy compliance, security safety, or production readiness.

In [ ]:
from azure.ai.evaluation import F1ScoreEvaluator, evaluate


def quality_floor_evaluator(response, ground_truth, **kwargs):
    response_terms = set(terms(response))
    reference_terms = set(terms(ground_truth))
    score = len(response_terms & reference_terms) / max(1, len(reference_terms))
    return {"score": score, "reason": "Normalized reference-term coverage."}


sdk_rows = live_results[[
    "request_id", "dataset_split", "workflow", "risk_tier", "task", "response", "ground_truth",
    "selected_model", "input_tokens", "output_tokens", "latency_ms",
]].to_dict(orient="records")

with TemporaryDirectory() as temporary_directory:
    data_path = Path(temporary_directory) / "live_router_eval.jsonl"
    output_path = Path(temporary_directory) / "live_router_eval_results.json"
    data_path.write_text(
        "".join(json.dumps(row, ensure_ascii=True) + "\n" for row in sdk_rows), encoding="utf-8",
    )
    captured_stdout = StringIO()
    with redirect_stdout(captured_stdout):
        evaluation_result = evaluate(
            data=str(data_path),
            evaluators={"domain_coverage": quality_floor_evaluator, "f1": F1ScoreEvaluator()},
            evaluator_config={
                "domain_coverage": {"column_mapping": {
                    "response": "${data.response}", "ground_truth": "${data.ground_truth}",
                }},
                "f1": {"column_mapping": {
                    "response": "${data.response}", "ground_truth": "${data.ground_truth}",
                }},
            },
            output_path=str(output_path),
        )

sdk_metrics = evaluation_result["metrics"]
print("Azure AI Evaluation metrics over 20 live Model Router responses:")
print(json.dumps(sdk_metrics, indent=2))

## 10. Export redacted live artifacts



The final cell writes a compact live-run summary and per-request operational measurements. Prompt and response text are omitted from the CSV export. Store production artifacts in governed storage and apply your organization’s retention policy.

In [ ]:
artifact_dir = Path("model_router_artifacts")
artifact_dir.mkdir(exist_ok=True)

redacted_columns = [
    "request_id", "dataset_split", "workflow", "risk_tier", "task", "selected_model",
    "input_tokens", "output_tokens", "latency_ms", "precision", "recall", "f1",
    "reference_coverage", "quality_pass",
]
live_results[redacted_columns].to_csv(
    artifact_dir / "redacted_live_router_results.csv", index=False,
)
(artifact_dir / "live_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8",
)
(artifact_dir / "evaluation_metrics.json").write_text(
    json.dumps(sdk_metrics, indent=2), encoding="utf-8",
)

print("Wrote redacted artifacts:")
for path in sorted(artifact_dir.iterdir()):
    print(" -", path.name)

## What you learned

1. A useful router scenario has a measurable tension: lower cost and latency without reducing policy compliance or account-security quality.
2. Core, challenge, and high-risk slices prevent an overall average from hiding operationally important failures.
3. Token and latency budgets can qualify a candidate, but only a controlled baseline comparison can establish improvement.
4. Passing this 20-case smoke test advances the router to Lab 2; it does not approve production promotion.

Previous: [Lab 0 - Setup](../lab-0-setup/SETUP.md) | [Workshop home](../README.md) | Next: [Lab 2 - Hill climbing](../lab-2-hill-climbing/README.md)